# Procesamiento de datos

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from gensim.models import Word2Vec, KeyedVectors
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize, word_tokenize
from transformers import BertTokenizer, BertModel, AutoTokenizer, AutoModel
import torch.nn.functional as F
import torch
import nltk
import numpy as np
from gensim.models.callbacks import CallbackAny2Vec
from sklearn.preprocessing import LabelEncoder

# Descargar datasets y combinarlos

In [18]:

# Diccionario para mapear tipos de falacia del segundo dataset a las clases del primero
fallacy_mapping = {
    # ad hominem
    "Ad Hominem": "ad hominem",
    "Circumstantial Ad Hominem": "ad hominem",
    "Tu Quoque": "ad hominem",
    "Abusive Ad Hominem": "ad hominem",
    "Guilt By Association": "ad hominem",
    "Argument From Commitment": "ad hominem",
    "Precedent Ad Hominem": "ad hominem",
    "Behavioral Ad Hominem": "ad hominem",
    "Ad Hominem Against a Witness at Trial": "ad hominem",

    # false dilemma
    "False Dichotomy": "false dilemma",
    "False Dilemma/Dichotomy": "false dilemma",
    "False dilemma": "false dilemma",

    # ad populum
    "Appeal to Popularity": "ad populum",
    "Bandwagon Fallacy": "ad populum",
    "Common Belief Fallacy": "ad populum",

    # equivocation
    "Equivocation": "equivocation",

    # fallacy of credibility
    "Argument from Authority": "fallacy of credibility",
    "Appeal to Authority": "fallacy of credibility",
    "Appeal to False Authority": "fallacy of credibility",
    "Argument from False Authority": "fallacy of credibility",
    "Appealing to an irrelevant authority": "fallacy of credibility",

    # false causality
    "Correlation does not imply causation": "false causality",
    "False cause": "false causality",
    "Post hoc ergo propter hoc": "false causality",
    "Cum hoc ergo propter hoc": "false causality",

    # intentional
    "Intentional Fallacy": "intentional",
    "Authorial Intent as Constraint": "intentional",

    # fallacy of logic / circular reasoning
    "Circular Reasoning": "circular reasoning",
    "Circular reasoning": "circular reasoning",
    "Fallacy of Logic": "fallacy of logic",
    "Begging the question": "fallacy of logic",
    "Begging the Question": "fallacy of logic",

    # appeal to emotion
    "Appeal to Emotion": "appeal to emotion",
    "Appeal to emotion": "appeal to emotion",
    "Appeal to Pity": "appeal to emotion",
    "Appeal to fear": "appeal to emotion",
    "Appeal to consequences": "appeal to emotion",

    # fallacy of relevance / extension
    "Fallacy of Extension": "fallacy of extension",
    "Fallacy of Relevance": "fallacy of relevance",
    "Red Herring": "fallacy of relevance",
    "Straw Man": "fallacy of relevance",
    "Straw man": "fallacy of relevance",
    "Strawman": "fallacy of relevance",

    # faulty generalization
    "Hasty Generalization": "faulty generalization",
    "Faulty Generalization ": "faulty generalization",
    "Hasty generalization": "faulty generalization",
    "Accident": "faulty generalization",
    "Generalization": "faulty generalization",
}

# Cargar datasets
dataset1 = load_dataset("tasksource/logical-fallacy")
dataset2 = load_dataset("MrOvkill/fallacies-fallacy-base")

train1, test1, dev1 = dataset1["train"], dataset1["test"], dataset1["dev"]

# Mapear clases del dataset2 a las del dataset1
def map_fallacy(example):
    mapped = fallacy_mapping.get(example["name"])
    return {"logical_fallacies": mapped}

dataset2_mapped = dataset2["train"].map(map_fallacy)
dataset2_mapped = dataset2_mapped.filter(lambda x: x["logical_fallacies"] is not None)

# Mantener solo columnas necesarias
dataset2_mapped = dataset2_mapped.remove_columns(
    [c for c in dataset2_mapped.column_names if c not in ["logical_fallacies", "example"]]
)

# Dividir dataset2 en train/dev/test (80/10/10)
data_array = dataset2_mapped["example"]
labels_array = dataset2_mapped["logical_fallacies"]

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    data_array, labels_array, test_size=0.2, stratify=labels_array, random_state=42
)
dev_texts, test_texts, dev_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

train2 = Dataset.from_dict({"example": train_texts, "logical_fallacies": train_labels})
dev2   = Dataset.from_dict({"example": dev_texts, "logical_fallacies": dev_labels})
test2  = Dataset.from_dict({"example": test_texts, "logical_fallacies": test_labels})

train2 = train2.rename_column("example", "source_article")
dev2   = dev2.rename_column("example", "source_article")
test2  = test2.rename_column("example", "source_article")

# Combinar datasets
train_combined = concatenate_datasets([train1, train2])
dev_combined   = concatenate_datasets([dev1, dev2])
test_combined  = concatenate_datasets([test1, test2])

all_labels = list(train_combined["logical_fallacies"]) + \
             list(dev_combined["logical_fallacies"]) + \
             list(test_combined["logical_fallacies"])

le = LabelEncoder()
le.fit(all_labels)

train_combined = train_combined.map(lambda x: {"logical_fallacies": int(le.transform([x["logical_fallacies"]])[0])})
dev_combined   = dev_combined.map(lambda x: {"logical_fallacies": int(le.transform([x["logical_fallacies"]])[0])})
test_combined  = test_combined.map(lambda x: {"logical_fallacies": int(le.transform([x["logical_fallacies"]])[0])})

dataset = DatasetDict({
    "train": train_combined,
    "dev": dev_combined,
    "test": test_combined
})

train = dataset["train"]
dev   = dataset["dev"]
test  = dataset["test"]

print(dataset)
print(f"Train examples: {len(dataset['train'])}")
print(f"Dev examples:   {len(dataset['dev'])}")
print(f"Test examples:  {len(dataset['test'])}")


DatasetDict({
    train: Dataset({
        features: ['config', 'source_article', 'logical_fallacies'],
        num_rows: 2901
    })
    dev: Dataset({
        features: ['config', 'source_article', 'logical_fallacies'],
        num_rows: 598
    })
    test: Dataset({
        features: ['config', 'source_article', 'logical_fallacies'],
        num_rows: 539
    })
})
Train examples: 2901
Dev examples:   598
Test examples:  539


In [19]:
# Descargar modelos de tonekizacion
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\marco\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\marco\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

# Tokenizacion + case-folding

In [20]:

def tokenize_flat(text):
    text = text.lower()
    sentences = sent_tokenize(text)
    tokens = []
    for sentence in sentences:
        tokens.extend(word_tokenize(sentence))
    return tokens

def preprocess_sentences(text):
    text = text.lower()
    sentences = sent_tokenize(text)
    tokenized_sentences = [word_tokenize(sentence) for sentence in sentences]
    return tokenized_sentences

train = train.map(lambda x: {"tokenized": tokenize_flat(x["source_article"])})
test  = test.map(lambda x: {"tokenized": tokenize_flat(x["source_article"])})
dev   = dev.map(lambda x: {"tokenized": tokenize_flat(x["source_article"])})

print(train[1]["source_article"])
print(train[1]["tokenized"])


The bigger a child's shoe size, the better the child's handwriting
['the', 'bigger', 'a', 'child', "'s", 'shoe', 'size', ',', 'the', 'better', 'the', 'child', "'s", 'handwriting']


# Embedding no contextual con Word2Vec (fine-tuneado)

In [21]:
# Selecciona qué modelo de Word2Vec usar
use_finetuned = True  # pon False para usar el modelo base

base_path = "./models/word2vec_base_fallacies.model"
ft_path   = "./models/word2vec_finetuned_fallacies.model"

if use_finetuned:
    print("Cargando Word2Vec fine-tuneado...")
    wv_model = Word2Vec.load(ft_path)
else:
    print("Cargando Word2Vec base...")
    wv_model = Word2Vec.load(base_path)

Cargando Word2Vec fine-tuneado...


In [22]:
class FallacyDatasetWord2Vec(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, vocab=None, embedding_model=None):
        self.dataset = hf_dataset
        self.vocab = vocab
        self.embedding_model = embedding_model
        self.tokenized_texts = hf_dataset["tokenized"]
        self.labels = hf_dataset["logical_fallacies"]

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        tokens = self.tokenized_texts[idx]
        label = self.labels[idx]

        if self.embedding_model is not None:
            embeddings = []
            for t in tokens:
                if t in self.embedding_model.wv:
                    embeddings.append(torch.tensor(self.embedding_model.wv[t], dtype=torch.float))
                else:
                    embeddings.append(torch.randn(self.embedding_model.vector_size))
            x = torch.stack(embeddings) if embeddings else torch.zeros(1, self.embedding_model.vector_size)
        elif self.vocab is not None:
            x = torch.tensor([self.vocab.get(t, self.vocab.get("<UNK>")) for t in tokens], dtype=torch.long)
        else:
            x = torch.tensor(tokens, dtype=torch.long)

        return x, torch.tensor(label, dtype=torch.long)


def collate_fn(batch):
    embeddings_list, labels_list = zip(*batch)
    lengths = [e.shape[0] for e in embeddings_list]
    max_len = max(lengths)
    embedding_dim = embeddings_list[0].shape[1]

    padded_embeddings = []
    for e in embeddings_list:
        if e.shape[0] < max_len:
            pad = torch.zeros(max_len - e.shape[0], embedding_dim)
            e_padded = torch.cat([e, pad], dim=0)
        else:
            e_padded = e
        padded_embeddings.append(e_padded)

    x_batch = torch.stack(padded_embeddings)
    y_batch = torch.tensor(labels_list)
    return x_batch, y_batch


# Clasificador CNN

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class CNN_NLP(nn.Module):
    def __init__(self, embed_dim=300, filter_sizes=[3,4,5], num_filters=[100,100,100], num_classes=12, dropout=0.5):
        super(CNN_NLP, self).__init__()
        self.conv1d_list = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters[i], kernel_size=filter_sizes[i])
            for i in range(len(filter_sizes))
        ])
        self.fc = nn.Linear(sum(num_filters), num_classes)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x_conv_list = [F.relu(conv1d(x)) for conv1d in self.conv1d_list]
        x_pool_list = [F.max_pool1d(conv, kernel_size=conv.shape[2]).squeeze(2) for conv in x_conv_list]
        x_fc = torch.cat(x_pool_list, dim=1)
        logits = self.fc(self.dropout(x_fc))
        return logits


In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score

# Configuración de hiperparámetros
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(set(train["logical_fallacies"]))
embed_dim = 300
batch_size = 16
num_epochs = 20
learning_rate = 1e-3
filter_sizes = [3, 4, 5]
num_filters = [100, 100, 100]
dropout = 0.5

# Inicializar modelo
cnn_model = CNN_NLP(embed_dim=embed_dim,
                    filter_sizes=filter_sizes,
                    num_filters=num_filters,
                    num_classes=num_classes,
                    dropout=dropout).to(device)

# Loss y optimizer
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

# DataLoaders
train_dataset = FallacyDatasetWord2Vec(train, embedding_model=wv_model)
dev_dataset   = FallacyDatasetWord2Vec(dev, embedding_model=wv_model)
train_loader  = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
dev_loader    = torch.utils.data.DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

# Función de accuracy
def compute_accuracy(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

# Training loop
for epoch in range(num_epochs):
    cnn_model.train()
    running_loss = 0.0
    running_acc = 0.0
    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = cnn_model(x_batch)
        loss = loss_func(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += (loss.item() - running_loss) / (batch_idx + 1)
        running_acc += (compute_accuracy(logits, y_batch) - running_acc) / (batch_idx + 1)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {running_loss:.4f} | Train Acc: {running_acc:.4f}")

    # Evaluación en dev
    cnn_model.eval()
    val_loss = 0.0
    val_acc = 0.0
    with torch.no_grad():
        for batch_idx, (x_batch, y_batch) in enumerate(dev_loader):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            logits = cnn_model(x_batch)
            loss = loss_func(logits, y_batch)
            val_loss += (loss.item() - val_loss) / (batch_idx + 1)
            val_acc += (compute_accuracy(logits, y_batch) - val_acc) / (batch_idx + 1)
    print(f"Validation Loss: {val_loss:.4f} | Validation Acc: {val_acc:.4f}")
    scheduler.step(val_loss)

# Evaluación final en test con métricas macro
test_dataset = FallacyDatasetWord2Vec(test, embedding_model=wv_model)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

cnn_model.eval()
test_loss = 0.0
all_preds = []
all_labels = []
with torch.no_grad():
    for batch_idx, (x_batch, y_batch) in enumerate(test_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        logits = cnn_model(x_batch)
        loss = loss_func(logits, y_batch)
        test_loss += (loss.item() - test_loss) / (batch_idx + 1)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

test_acc = (sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels))
test_precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
test_recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
test_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")
print(f"Test Precision (macro): {test_precision:.4f} | Recall (macro): {test_recall:.4f} | F1 (macro): {test_f1:.4f}")

Epoch 1/20 | Train Loss: 2.3453 | Train Acc: 0.2295
Validation Loss: 2.1887 | Validation Acc: 0.2961
Epoch 2/20 | Train Loss: 2.0114 | Train Acc: 0.3679
Validation Loss: 2.0345 | Validation Acc: 0.3438
Epoch 3/20 | Train Loss: 1.7801 | Train Acc: 0.4343
Validation Loss: 1.9626 | Validation Acc: 0.3651
Epoch 4/20 | Train Loss: 1.5928 | Train Acc: 0.5052
Validation Loss: 1.9421 | Validation Acc: 0.3882
Epoch 5/20 | Train Loss: 1.3985 | Train Acc: 0.5699
Validation Loss: 1.9334 | Validation Acc: 0.3728
Epoch 6/20 | Train Loss: 1.2391 | Train Acc: 0.6255
Validation Loss: 1.9095 | Validation Acc: 0.3734
Epoch 7/20 | Train Loss: 1.0840 | Train Acc: 0.6724
Validation Loss: 1.8944 | Validation Acc: 0.4024
Epoch 8/20 | Train Loss: 0.9387 | Train Acc: 0.7201
Validation Loss: 1.8938 | Validation Acc: 0.4172
Epoch 9/20 | Train Loss: 0.8211 | Train Acc: 0.7609
Validation Loss: 1.9559 | Validation Acc: 0.3991
Epoch 10/20 | Train Loss: 0.7652 | Train Acc: 0.7802
Validation Loss: 1.9142 | Validation A

In [25]:
import pandas as pd, os

run_label = "w2v_base" if not use_finetuned else "w2v_finetuned"
results_row = {
    "run": run_label,
    "epochs": num_epochs,
    "val_loss": val_loss,
    "val_acc": val_acc,
    "test_loss": test_loss,
    "test_acc": test_acc,
}

out_path = "experiments/cnn_word2vec_runs.csv"
exists = os.path.exists(out_path)
pd.DataFrame([results_row]).to_csv(out_path, mode="a" if exists else "w", header=not exists, index=False)
print(f"Guardado en {out_path}")

Guardado en experiments/cnn_word2vec_runs.csv


# Embedding contextual con BERT

In [26]:

# Cargar modelo y tokenizer BERT preentrenado
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

def get_bert_cls_embeddings(sentences, batch_size=16, max_length=128, device='cpu'):

    # Devuelve la representación del token [CLS] para cada texto
    model.to(device)
    cls_embeddings = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        batch_texts = [" ".join(s) for s in batch]
        encoded = tokenizer(
            batch_texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=max_length,
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)
            # Tomamos la representación del token [CLS] (posición 0) del último hidden state
            batch_cls = outputs.last_hidden_state[:, 0, :]
            for emb in batch_cls:
                # Movemos a CPU para facilitar cálculo posterior
                cls_embeddings.append(emb.cpu())

    return cls_embeddings

# Generar embeddings CLS para train/test/dev
train_sentences = list(train["tokenized"])
test_sentences  = list(test["tokenized"])
dev_sentences   = list(dev["tokenized"])

device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_embeddings = get_bert_cls_embeddings(train_sentences, device=device)
test_embeddings  = get_bert_cls_embeddings(test_sentences, device=device)
dev_embeddings   = get_bert_cls_embeddings(dev_sentences, device=device)

print(f"Ejemplo: embedding CLS train[0] shape: {train_embeddings[0].shape}")

# Ejemplo de similitud coseno 
sim = F.cosine_similarity(train_embeddings[0], train_embeddings[1], dim=0)
print(f"Similitud coseno entre train[0] y train[1]: {sim.item():.4f}")


Ejemplo: embedding CLS train[0] shape: torch.Size([768])
Similitud coseno entre train[0] y train[1]: 0.8243


In [27]:

class FallacyDatasetBERT(torch.utils.data.Dataset):
    def __init__(self, embeddings, labels):
        """
        embeddings: lista de tensores torch (CLS embeddings)
        labels: lista de enteros (labels)
        """
        self.embeddings = embeddings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.embeddings[idx]
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

def collate_fn_bert(batch):
    x_batch, y_batch = zip(*batch)
    x_batch = torch.stack(x_batch)
    y_batch = torch.tensor(y_batch)
    return x_batch, y_batch


# Token CLS BERT + MLP

In [28]:
class MLP_BERT(nn.Module):
    def __init__(self, embed_dim=768, hidden_dim=512, num_classes=12, dropout=0.5):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        logits = self.fc3(x)
        return logits

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score

# Configuración de hiperparámetros
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(set(train["logical_fallacies"]))
embed_dim = 768
hidden_dim = 512
batch_size = 16
num_epochs = 40
learning_rate = 1e-3
dropout = 0.5

# Inicializar modelo MLP
mlp_model = MLP_BERT(embed_dim=embed_dim,
                     hidden_dim=hidden_dim,
                     num_classes=num_classes,
                     dropout=dropout).to(device)

# Loss y optimizer
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

# DataLoaders
train_dataset = FallacyDatasetBERT(train_embeddings, list(train["logical_fallacies"]))
dev_dataset   = FallacyDatasetBERT(dev_embeddings, list(dev["logical_fallacies"]))

train_loader  = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn_bert)
dev_loader    = torch.utils.data.DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_bert)

# Función de accuracy
def compute_accuracy(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

# Training loop
for epoch in range(num_epochs):
    mlp_model.train()
    running_loss = 0.0
    running_acc = 0.0
    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = mlp_model(x_batch)
        loss = loss_func(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += (loss.item() - running_loss) / (batch_idx + 1)
        running_acc += (compute_accuracy(logits, y_batch) - running_acc) / (batch_idx + 1)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {running_loss:.4f} | Train Acc: {running_acc:.4f}")

    # Evaluación en dev
    mlp_model.eval()
    val_loss = 0.0
    val_acc = 0.0
    with torch.no_grad():
        for batch_idx, (x_batch, y_batch) in enumerate(dev_loader):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            logits = mlp_model(x_batch)
            loss = loss_func(logits, y_batch)
            val_loss += (loss.item() - val_loss) / (batch_idx + 1)
            val_acc += (compute_accuracy(logits, y_batch) - val_acc) / (batch_idx + 1)
    print(f"Validation Loss: {val_loss:.4f} | Validation Acc: {val_acc:.4f}")
    scheduler.step(val_loss)

# Evaluación final en test con métricas macro
test_dataset = FallacyDatasetBERT(test_embeddings, list(test["logical_fallacies"]))
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_bert)

mlp_model.eval()
test_loss = 0.0
all_preds = []
all_labels = []
with torch.no_grad():
    for batch_idx, (x_batch, y_batch) in enumerate(test_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        logits = mlp_model(x_batch)
        loss = loss_func(logits, y_batch)
        test_loss += (loss.item() - test_loss) / (batch_idx + 1)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

test_acc = (sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels))
test_precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
test_recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
test_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")
print(f"Test Precision (macro): {test_precision:.4f} | Recall (macro): {test_recall:.4f} | F1 (macro): {test_f1:.4f}")

Epoch 1/40 | Train Loss: 2.4400 | Train Acc: 0.1854
Validation Loss: 2.2756 | Validation Acc: 0.2478
Epoch 2/40 | Train Loss: 2.2741 | Train Acc: 0.2453
Validation Loss: 2.1674 | Validation Acc: 0.3185
Epoch 3/40 | Train Loss: 2.1522 | Train Acc: 0.2947
Validation Loss: 2.1279 | Validation Acc: 0.2961
Epoch 4/40 | Train Loss: 2.0541 | Train Acc: 0.3280
Validation Loss: 1.9704 | Validation Acc: 0.3810
Epoch 5/40 | Train Loss: 1.9473 | Train Acc: 0.3679
Validation Loss: 1.9401 | Validation Acc: 0.3766
Epoch 6/40 | Train Loss: 1.8700 | Train Acc: 0.3889
Validation Loss: 1.9356 | Validation Acc: 0.3734
Epoch 7/40 | Train Loss: 1.8299 | Train Acc: 0.4098
Validation Loss: 1.9255 | Validation Acc: 0.3832
Epoch 8/40 | Train Loss: 1.7482 | Train Acc: 0.4238
Validation Loss: 1.8733 | Validation Acc: 0.4057
Epoch 9/40 | Train Loss: 1.7040 | Train Acc: 0.4558
Validation Loss: 1.8719 | Validation Acc: 0.4139
Epoch 10/40 | Train Loss: 1.6713 | Train Acc: 0.4479
Validation Loss: 1.8576 | Validation A

# Embedding contextual Distil-RoBERTa

In [30]:

tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")
model = AutoModel.from_pretrained("distilroberta-base")
model.eval()

def get_distilroberta_cls_embeddings(sentences, batch_size=16, max_length=128, device="cpu"):
    model.to(device)
    cls_embeddings = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        batch_texts = [" ".join(s) for s in batch]

        encoded = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoded)
            batch_cls = outputs.last_hidden_state[:, 0, :]

        for emb in batch_cls:
            cls_embeddings.append(emb.cpu())

    return cls_embeddings

train_embeddings = get_distilroberta_cls_embeddings(list(train["tokenized"]), device=device)
dev_embeddings   = get_distilroberta_cls_embeddings(list(dev["tokenized"]), device=device)
test_embeddings  = get_distilroberta_cls_embeddings(list(test["tokenized"]), device=device)

print(train_embeddings[0].shape)

c:\Users\marco\Documents\Deusto\procesamiento_del_lenguaje_natural\fallacy-classification\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\marco\.cache\huggingface\hub\models--distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' 

torch.Size([768])


In [31]:
class FallacyDatasetDR(torch.utils.data.Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = embeddings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], torch.tensor(self.labels[idx], dtype=torch.long)

def collate_fn_dr(batch):
    xs, ys = zip(*batch)
    return torch.stack(xs), torch.tensor(ys)


In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score

# Configuración de hiperparámetros
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(set(train["logical_fallacies"]))
embed_dim = 768
hidden_dim = 512
batch_size = 16
num_epochs = 40
learning_rate = 1e-3
dropout = 0.5

# Inicializar modelo MLP
mlp_model = MLP_BERT(embed_dim=embed_dim,
                     hidden_dim=hidden_dim,
                     num_classes=num_classes,
                     dropout=dropout).to(device)

# Loss y optimizer
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

# DataLoaders
train_dataset = FallacyDatasetDR(train_embeddings, list(train["logical_fallacies"]))
dev_dataset   = FallacyDatasetDR(dev_embeddings, list(dev["logical_fallacies"]))

train_loader  = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn_dr)
dev_loader    = torch.utils.data.DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_dr)

# Función de accuracy
def compute_accuracy(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

# Training loop
for epoch in range(num_epochs):
    mlp_model.train()
    running_loss = 0.0
    running_acc = 0.0
    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = mlp_model(x_batch)
        loss = loss_func(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += (loss.item() - running_loss) / (batch_idx + 1)
        running_acc += (compute_accuracy(logits, y_batch) - running_acc) / (batch_idx + 1)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {running_loss:.4f} | Train Acc: {running_acc:.4f}")

    # Evaluación en dev
    mlp_model.eval()
    val_loss = 0.0
    val_acc = 0.0
    with torch.no_grad():
        for batch_idx, (x_batch, y_batch) in enumerate(dev_loader):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            logits = mlp_model(x_batch)
            loss = loss_func(logits, y_batch)
            val_loss += (loss.item() - val_loss) / (batch_idx + 1)
            val_acc += (compute_accuracy(logits, y_batch) - val_acc) / (batch_idx + 1)
    print(f"Validation Loss: {val_loss:.4f} | Validation Acc: {val_acc:.4f}")
    scheduler.step(val_loss)

# Evaluación final en test con métricas macro
test_dataset = FallacyDatasetDR(test_embeddings, list(test["logical_fallacies"]))
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_dr)

mlp_model.eval()
test_loss = 0.0
all_preds = []
all_labels = []
with torch.no_grad():
    for batch_idx, (x_batch, y_batch) in enumerate(test_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        logits = mlp_model(x_batch)
        loss = loss_func(logits, y_batch)
        test_loss += (loss.item() - test_loss) / (batch_idx + 1)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

test_acc = (sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels))
test_precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
test_recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
test_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")
print(f"Test Precision (macro): {test_precision:.4f} | Recall (macro): {test_recall:.4f} | F1 (macro): {test_f1:.4f}")

Epoch 1/40 | Train Loss: 2.5008 | Train Acc: 0.1391
Validation Loss: 2.4658 | Validation Acc: 0.2204
Epoch 2/40 | Train Loss: 2.4380 | Train Acc: 0.1776
Validation Loss: 2.3580 | Validation Acc: 0.2681
Epoch 3/40 | Train Loss: 2.3650 | Train Acc: 0.2242
Validation Loss: 2.2727 | Validation Acc: 0.2577
Epoch 4/40 | Train Loss: 2.2741 | Train Acc: 0.2519
Validation Loss: 2.1821 | Validation Acc: 0.2966
Epoch 5/40 | Train Loss: 2.1758 | Train Acc: 0.2786
Validation Loss: 2.1741 | Validation Acc: 0.3125
Epoch 6/40 | Train Loss: 2.1456 | Train Acc: 0.3036
Validation Loss: 2.1657 | Validation Acc: 0.2933
Epoch 7/40 | Train Loss: 2.1034 | Train Acc: 0.3144
Validation Loss: 2.0745 | Validation Acc: 0.3213
Epoch 8/40 | Train Loss: 2.0439 | Train Acc: 0.3259
Validation Loss: 2.0034 | Validation Acc: 0.3437
Epoch 9/40 | Train Loss: 2.0138 | Train Acc: 0.3504
Validation Loss: 2.0048 | Validation Acc: 0.3470
Epoch 10/40 | Train Loss: 1.9950 | Train Acc: 0.3556
Validation Loss: 1.9758 | Validation A

# LSTM bidireccional + atencion

In [33]:

class BiLSTM_Attention(nn.Module):
    def __init__(self, embed_dim=300, hidden_dim=256, num_layers=1, num_classes=12, dropout=0.5):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = True
        self.lstm = nn.LSTM(input_size=embed_dim,
                            hidden_size=hidden_dim,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=self.bidirectional,
                            dropout=dropout if num_layers > 1 else 0)

        self.attention_fc = nn.Linear(hidden_dim * 2, 1)

        # Fully connected final
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)

        attn_weights = F.softmax(self.attention_fc(lstm_out), dim=1)
        attn_applied = torch.sum(lstm_out * attn_weights, dim=1)

        logits = self.fc(self.dropout(attn_applied))
        return logits


In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score

# Configuración de hiperparámetros
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(set(train["logical_fallacies"]))
embed_dim = 300
hidden_dim = 256
num_layers = 2
dropout = 0.5
batch_size = 16
num_epochs = 20
learning_rate = 1e-3

# Inicializar modelo
model = BiLSTM_Attention(embed_dim=embed_dim,
                         hidden_dim=hidden_dim,
                         num_layers=num_layers,
                         num_classes=num_classes,
                         dropout=dropout).to(device)

# DataLoaders
train_dataset = FallacyDatasetWord2Vec(train, embedding_model=wv_model)
dev_dataset   = FallacyDatasetWord2Vec(dev, embedding_model=wv_model)
test_dataset  = FallacyDatasetWord2Vec(test, embedding_model=wv_model)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
dev_loader   = torch.utils.data.DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

# Loss y optimizer
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

# Función de accuracy
def compute_accuracy(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

# Loop de entrenamiento
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    running_acc = 0.0

    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = loss_func(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += (loss.item() - running_loss) / (batch_idx + 1)
        running_acc += (compute_accuracy(logits, y_batch) - running_acc) / (batch_idx + 1)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {running_loss:.4f} | Train Acc: {running_acc:.4f}")

    # Evaluación en dev
    model.eval()
    val_loss = 0.0
    val_acc = 0.0
    with torch.no_grad():
        for batch_idx, (x_batch, y_batch) in enumerate(dev_loader):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            logits = model(x_batch)
            loss = loss_func(logits, y_batch)
            val_loss += (loss.item() - val_loss) / (batch_idx + 1)
            val_acc += (compute_accuracy(logits, y_batch) - val_acc) / (batch_idx + 1)

    print(f"Validation Loss: {val_loss:.4f} | Validation Acc: {val_acc:.4f}")
    scheduler.step(val_loss)

# Evaluación final en test con métricas macro
model.eval()
test_loss = 0.0
all_preds = []
all_labels = []
with torch.no_grad():
    for batch_idx, (x_batch, y_batch) in enumerate(test_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        logits = model(x_batch)
        loss = loss_func(logits, y_batch)
        test_loss += (loss.item() - test_loss) / (batch_idx + 1)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

test_acc = (sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels))
test_precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
test_recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
test_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")
print(f"Test Precision (macro): {test_precision:.4f} | Recall (macro): {test_recall:.4f} | F1 (macro): {test_f1:.4f}")

Epoch 1/20 | Train Loss: 2.4378 | Train Acc: 0.1762
Validation Loss: 2.2790 | Validation Acc: 0.2516
Epoch 2/20 | Train Loss: 2.1849 | Train Acc: 0.2784
Validation Loss: 2.1752 | Validation Acc: 0.3032
Epoch 3/20 | Train Loss: 1.9978 | Train Acc: 0.3497
Validation Loss: 2.0635 | Validation Acc: 0.3328
Epoch 4/20 | Train Loss: 1.8724 | Train Acc: 0.4095
Validation Loss: 1.9802 | Validation Acc: 0.3591
Epoch 5/20 | Train Loss: 1.7304 | Train Acc: 0.4462
Validation Loss: 1.9168 | Validation Acc: 0.3723
Epoch 6/20 | Train Loss: 1.5992 | Train Acc: 0.4953
Validation Loss: 1.9449 | Validation Acc: 0.3914
Epoch 7/20 | Train Loss: 1.4688 | Train Acc: 0.5382
Validation Loss: 1.9656 | Validation Acc: 0.3947
Epoch 8/20 | Train Loss: 1.2205 | Train Acc: 0.6151
Validation Loss: 1.9651 | Validation Acc: 0.4156
Epoch 9/20 | Train Loss: 1.0600 | Train Acc: 0.6671
Validation Loss: 2.1222 | Validation Acc: 0.3712
Epoch 10/20 | Train Loss: 0.8742 | Train Acc: 0.7352
Validation Loss: 2.0964 | Validation A